For the iterative workflow: Here the post will be generated by the llm and then evaluated by another llm . This second llm will suggest the improvements and decide whether post is good enough and fulfills the strict criteria set by the user on loop. This doesnot have the human in the loop but it is needed on the actual system. After the post is approved it will hit api and post it Else more optimization in the loop 

In [3]:
from langgraph.graph import StateGraph,START, END
from typing import Literal,TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

It is best to use the different llm for the generation , evaluation . LLM should be chosen based on its forte. Since some are good at evaulation, mathematics, some on the literature . So Using llm with the best capabilities is better 

In [4]:

generator_llm =  ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
evaluator_llm =  ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
optimizer_llm =  ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [2]:
# state 

class XState(TypedDict):

    topic: str
    tweet: str
    evaluation: Literal['approved','needs_improvement']
    feedback: str
    iterations: int 
    max_iterations:int  # to avoid the infite loop 
    

In [ ]:
def generate_tweet(state:XState):

    # prompt 
        messages = [
                        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
                        HumanMessage(content=f"""
                                Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

                                Rules:
                                - Do NOT use question-answer format.
                                - Max 280 characters.
                                - Use observational humor, irony, sarcasm, or cultural references.
                                - Think in meme logic, punchlines, or relatable takes.
                                - Use simple, day to day english
                                """

                                )
                    ]

        # send generator_llm
        response = generator_llm.invoke(messages).content

        # return response
        return {'tweet': response, 'tweet_history': [response]}

In [ ]:
# graph

graph = StateGraph(XState)

graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)